In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import plotly.express as px
import datetime as dt
from datetime import timedelta

In [22]:
TIMEFRAME_DAYS = {
    "1D": 1,
}

In [12]:
current_date = dt.datetime.now().date()
tickers = ['QQQ', 'SPY', 'AAPL', 'JPY=X', 'EUR=X']

date_5_years_ago = current_date- timedelta(days=1850)
df = yf.download(tickers,start =date_5_years_ago )['Close']
#df = df.ffill()

df

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_26377/729718702.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers,start =date_5_years_ago )['Close']
[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,EUR=X,JPY=X,QQQ,SPY
Date,,,,,
2021-03-29,118.233322,0.848320,109.791000,306.568512,369.651764
2021-03-30,116.782089,0.849545,109.835999,305.025543,368.671051
2021-03-31,118.973557,0.852980,110.305000,309.693420,370.165466
2021-04-01,119.801430,0.852600,110.750000,314.972504,374.162903
2021-04-02,NaN,0.848900,110.612999,NaN,NaN
...,...,...,...,...,...
2026-04-15,266.429993,0.847500,158.792007,637.400024,699.940002
2026-04-16,263.399994,0.846800,158.809006,640.469971,701.659973
2026-04-17,270.230011,0.848700,159.195007,648.849976,710.140015


In [25]:
returns_dict = {tf: {} for tf in TIMEFRAME_DAYS}
for ticker in df.columns:
    print(f'ticker {ticker}')
    last_idx = df[ticker].last_valid_index()
    #print(last_idx)
    if last_idx is None:
        continue
    last_pos = df.index.get_loc(last_idx)
    #print(last_pos)
    last_price = df[ticker].iloc[last_pos]
    print(f'last price {last_pos}')
    print(f'last price {last_price}')

    for tf, days in TIMEFRAME_DAYS.items():
            prev_pos = last_pos - days
    
            print(f'prev pos {prev_pos}')
            if prev_pos < 0:
                continue
            prev_price = df[ticker].iloc[prev_pos]
            print(f'prev price {prev_price}')
            
            if pd.notna(prev_price) and prev_price != 0:
                returns_dict[tf][ticker] = (last_price - prev_price) / prev_price * 100



ticker AAPL
last price 1316
last price 273.04998779296875
prev pos 1315
prev price 270.2300109863281
ticker EUR=X
last price 1317
last price 0.8485000133514404
prev pos 1316
prev price 0.8517000079154968
ticker JPY=X
last price 1317
last price 158.9080047607422
prev pos 1316
prev price 159.16099548339844
ticker QQQ
last price 1316
last price 646.7899780273438
prev pos 1315
prev price 648.8499755859375
ticker SPY
last price 1316
last price 708.719970703125
prev pos 1315
prev price 710.1400146484375


In [ ]:
# Per-ticker anchor on each ticker's last non-NaN row instead of df.iloc[-1].
# During US market hours df's last row has equities NaN but 24/7 FX/crypto
# populated. Using a shared iloc[-1] + ffill there gives equities' 1D return = 0
# (today ffills to yesterday's close). Anchoring per ticker means equities
# compute returns off yesterday (their true last settled bar), FX/crypto off
# today, and the treemap shows real moves for everything.

    for tf, days in TIMEFRAME_DAYS.items():
        prev_pos = last_pos - days
        if prev_pos < 0:
            continue
        prev_price = df[ticker].iloc[prev_pos]
        if pd.notna(prev_price) and prev_price != 0:
            returns_dict[tf][ticker] = (last_price - prev_price) / prev_price * 100

returns_df = pd.DataFrame(returns_dict)
returns_df

,1D,1W,1M,3M,1Y,2Y,5Y
AAPL,1.043547,5.343355,10.105239,10.359574,29.842351,50.779762,115.878384
EUR=X,-0.375719,-0.141222,-1.912049,-0.954845,-3.362107,-8.812462,2.898379
JPY=X,-0.158953,-0.192194,-0.204723,0.285260,11.671909,2.244898,44.351591
QQQ,-0.317484,4.761976,11.260963,5.083001,36.672916,48.591873,94.734491
SPY,-0.199967,3.296895,9.274244,3.684781,29.319589,40.367792,78.763362


In [ ]:
current_date = dt.datetime.now().date()
date_5_years_ago = current_date - timedelta(days=1850)

# Mix of asset classes so the 70%-complete trim threshold has signal:
#   3 equities (US session, weekdays 09:30-16:00 ET) + 2 FX (~24/7).
# During US market hours: equities NaN, FX populated → 2/5 = 40% < 70% → trim.
# Post-close / weekend: depends on day, but the trim condition fires cleanly.

df = yf.download(tickers, start=date_5_years_ago)['Close']
df.tail()

/var/folders/hs/v5c0lrnj61z3tp745vdhh0m40000gn/T/ipykernel_26377/766493960.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, start=date_5_years_ago)['Close']
[*********************100%***********************]  5 of 5 completed


Ticker,AAPL,EUR=X,JPY=X,QQQ,SPY
Date,,,,,
2026-04-15,266.429993,0.8475,158.792007,637.400024,699.940002
2026-04-16,263.399994,0.8468,158.809006,640.469971,701.659973
2026-04-17,270.230011,0.8487,159.195007,648.849976,710.140015
2026-04-20,273.049988,0.8517,159.160995,646.789978,708.719971
2026-04-21,NaN,0.8484,158.884995,NaN,NaN


In [10]:
returns_dict = {}
last_price = df.iloc[-1]
for tf, days in TIMEFRAME_DAYS.items():
    return_for_tf = ((last_price - df.iloc[-days - 1]) / df.iloc[-days - 1]) * 100
    returns_dict[tf] = return_for_tf

returns_df = pd.DataFrame(returns_dict)
returns_df

,1D,1W,1M,3M,1Y,2Y,5Y
Ticker,,,,,,,
AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EUR=X,-0.387462,-0.152993,-1.923611,-0.966520,-3.373499,-8.823211,2.886250
JPY=X,-0.173410,-0.206646,-0.219174,0.270738,11.655738,2.230093,44.330688
QQQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SPY,NaN,NaN,NaN,NaN,NaN,NaN,NaN
